# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. All entities (record sets, fields, and columns) are referenced **exclusively by their `@id`** as required by the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset and its metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', None)}")

## 2. Data Overview
Review available record sets and their fields. All entities are referenced by their `@id`.

**Tip:** You can inspect the structure of record sets, fields, and columns before deciding which data to extract.

In [ ]:
import pprint

# List all available record sets by `@id` and print their associated fields by `@id`
all_record_sets = list(dataset.metadata.record_sets)
print(f"Record sets found (by @id): {[rs.id for rs in all_record_sets]}")

# Print fields for each record set (by @id)
for rs in all_record_sets:
    print(f"\nRecordSet @id: {rs.id}")
    fields = getattr(rs, 'fields', [])
    columns = getattr(rs, 'columns', [])
    print("  Fields @id list:")
    for field in fields:
        print(f"     - {field.id}")
    if columns:
        print("  Columns @id list:")
        for col in columns:
            print(f"     - {col.id}")

## 3. Data Extraction
Load data from selected record set(s) into DataFrames for analysis.

### Instructions
- Use the `@id` of the desired record set(s) from the previous cell.
- All field and column accesses use their `@id` as per Croissant's specification.

In [ ]:
# Select the main record set(s) to load
# For this dataset, most analysis will focus on the main tabular data record set

# You may update the list below based on output from the previous cell
main_record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034-table1'  # Example @id for illustration
# For demonstration: dynamically find the main data table
main_record_sets = [rs for rs in dataset.metadata.record_sets if hasattr(rs, 'columns') and rs.columns]
record_set_ids = [rs.id for rs in main_record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show available columns (by Field/Column @id) for the first loaded table
first_rs_id = record_set_ids[0] if record_set_ids else None
if first_rs_id:
    print(f"\nColumns in RecordSet '{first_rs_id}':")
    print(list(dataframes[first_rs_id].columns))
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps: filter, normalize, and group by key fields. Customize the field `@id`s to match those available in your selected record set.

In [ ]:
# Example EDA: Filter and normalize a numeric field

# Substitute with an appropriate numeric field @id. Here we try to auto-detect a numeric column.
df = dataframes[first_rs_id]
numeric_field_id = None
for col in df.columns:
    # Heuristically select a numeric column
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is not None:
    print(f"Using numeric field @id: {numeric_field_id}")
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered rows where {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} records")

    norm_colname = f"{numeric_field_id}_normalized"
    filtered_df[norm_colname] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, norm_colname]].head())

    # Attempt to find a categorical field for grouping
    group_field_id = None
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field_id:
            group_field_id = col
            break
    if group_field_id is not None:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['count', 'mean', 'std'])
        print(f"\nGrouped by {group_field_id}:")
        print(grouped.head())
else:
    print("No numeric field found for processing.")

## 5. Visualization
Visualize the distribution of the selected numeric field, and if available, how it varies with respect to a group (categorical) field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
- The dataset provides mass spectrometry protein abundance and modification data for extracellular vesicles from human mast cells.
- Using `mlcroissant`, you explored record set structure, dynamically loaded data via record set `@id`, and performed basic EDA referencing only Croissant schema `@id` entities.
- Further biological or clinical analyses can be performed using the extracted DataFrames according to FAIR and reproducible research standards.

**Note:** For more targeted analysis, revisit section 2 to explore individual record sets, field and column `@id`s for your specific research question.